# Medical QA — Fine-tuning Mistral-7B on PubMedQA

QLoRA fine-tuning of Mistral-7B-Instruct to answer biomedical research questions. Given a research question and supporting abstract, the model outputs yes, no, or maybe with a grounded explanation.


**Model** : mistral-7b-instruct-v0.2 (4-bit, via Unsloth)

**Dataset** : PubMedQA `pqa_labeled` — 1,000 human-annotated biomedical Q&A pairs

**Method** : QLoRA — 4-bit base + 16-bit LoRA adapters (~0.6% trainable parameters)

**Hardware** : Google Colab T4 (free tier)

**Runtime -> Change runtime type -> T4 GPU before running.**

## 1. Dependencies

In [1]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps unsloth
!pip install pyarrow --upgrade -q
!pip install datasets trl evaluate

print("Done. Restart runtime now, then run from cell 2 onwards.")

## 2. Config

In [2]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import json

print(f"PyTorch : {torch.__version__}")
print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

# Model
MODEL_NAME   = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"
MAX_SEQ_LEN  = 512
LOAD_IN_4BIT = True

# LoRA — rank=8, alpha=16 tuned for T4 memory constraints
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

# Training — effective batch size = 1 * 4 = 4
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
LR_SCHEDULER = "cosine"
OUTPUT_DIR = "./pubmedqa-checkpoints"

# Dataset
DATASET_NAME = "qiaojin/PubMedQA"
DATASET_SPLIT = "pqa_labeled"

# HuggingFace repo — update with your username
HF_REPO_NAME = "ShardulJ/mistral-pubmedqa-qlora"

print("\nConfig loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch : 2.10.0+cu128
GPU : Tesla T4
VRAM : 15.6 GB

Config loaded.


## 3. Dataset

PubMedQA samples are formatted into Mistral's `[INST]...[/INST]` instruction template. The prompt is masked during training — loss is computed on the response tokens only.

In [3]:
raw_dataset = load_dataset(DATASET_NAME, DATASET_SPLIT)
print("Loaded:", raw_dataset)

SYSTEM_PROMPT = (
    "You are a biomedical research assistant. "
    "Given a research question and relevant abstract context, "
    "answer with 'yes', 'no', or 'maybe', followed by a concise explanation "
    "grounded in the provided context."
)

def format_sample(example):
    sentences = example["context"]["contexts"][:5]
    context = "\n".join(sentences)
    user_message = SYSTEM_PROMPT + "\n\nQuestion: " + example["question"] + "\n\nContext:\n" + context
    decision = example["final_decision"].upper()
    assistant_response = decision + ": " + example["long_answer"]
    text = f"<s>[INST] {user_message} [/INST] {assistant_response}</s>"
    return {"text": text}

dataset = raw_dataset["train"].map(format_sample)

train_dataset = dataset.select(range(0, 800))
val_dataset = dataset.select(range(800, 900))
test_dataset = dataset.select(range(900, 1000))

# Filter samples exceeding MAX_SEQ_LEN to avoid batch size mismatches during training
def filter_long_samples(example):
    tokens = tokenizer(example["text"], truncation=False)
    return len(tokens["input_ids"]) <= MAX_SEQ_LEN

print(f"\nSplit — Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded: DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


Split — Train: 800 | Val: 100 | Test: 100


## 4. Model & LoRA

Base weights are quantized to 4-bit (NF4) and frozen. LoRA injects trainable rank-8 adapter matrices into the attention projections — approximately 0.6% of total parameters are trained.

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

# Filter long samples now that tokenizer is available
train_dataset = train_dataset.filter(filter_long_samples)
val_dataset   = val_dataset.filter(filter_long_samples)

total, trainable = count_parameters(model)
print(f"Total params : {total:,}")
print(f"Trainable params : {trainable:,}")
print(f"Trainable % : {100 * trainable / total:.2f}%")

==((====))==  Unsloth 2026.4.6: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.6 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Filter:   0%|          | 0/800 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

Total params     : 3,755,479,040
Trainable params : 3,407,872
Trainable %      : 0.09%


## 5. Training

In [5]:
training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    num_train_epochs = NUM_TRAIN_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    warmup_ratio = WARMUP_RATIO,
    learning_rate = LEARNING_RATE,
    lr_scheduler_type = LR_SCHEDULER,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    report_to = "none",
    seed = 42,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    packing = False,
)

trainer_stats = trainer.train()

print(f"\n Training complete")
print(f"Steps : {trainer_stats.global_step}")
print(f"Training loss : {trainer_stats.training_loss:.4f}")
print(f"Runtime : {trainer_stats.metrics['train_runtime'] / 60:.1f} min")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/492 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/47 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 492 | Num Epochs = 3 | Total steps = 369
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 3,407,872 of 7,245,139,968 (0.05% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,1.333803,1.320618
2,1.386303,1.320788
3,1.272560,1.326392


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


 Training complete
Steps : 369
Training loss : 1.3908
Runtime : 26.5 min


## 6. Evaluation

In [6]:
FastLanguageModel.for_inference(model)

def build_inference_prompt(question, context_sentences):
    context = "\n".join(context_sentences)
    user_message = SYSTEM_PROMPT + "\n\nQuestion: " + question + "\n\nContext:\n" + context
    return "<s>[INST] " + user_message + " [/INST]"

def generate_answer(model, tokenizer, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.1, do_sample=True)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

results = []
n_eval  = 10

print("Evaluating on test samples...\n")
print("=" * 80)

for i, example in enumerate(test_dataset.select(range(n_eval))):
    question = example["question"]
    context = example["context"]["contexts"][:5]
    label = example["final_decision"]

    prompt = build_inference_prompt(question, context)
    response = generate_answer(model, tokenizer, prompt)
    predicted = response.strip().split()[0].lower().rstrip(".:,")
    correct = predicted == label

    results.append({"question": question, "label": label, "predicted": predicted,
                    "response": response, "correct": correct})

    print(f"[{i+1}] {question[:80]}")
    print(f"Label : {label}")
    print(f"Predicted : {predicted}  {'✓' if correct else '✗'}")
    print(f"Response : {response[:200]}")
    print()

accuracy = sum(r["correct"] for r in results) / len(results)
print("=" * 80)
print(f"\nAccuracy : {accuracy:.1%}  (random baseline: 33.3%)")

with open("eval_results.json", "w") as f:
    json.dump({"accuracy": accuracy, "n_eval": n_eval, "results": results}, f, indent=2)
print("Results saved to eval_results.json")

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating on test samples...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

[1] Internal derangement of the temporomandibular joint: is there still a place for 
Label : no
Predicted : no  ✓
Response : NO: The results of this study suggest that the use of ultrasound in the diagnosis of internal derangements of the temporomandibular joint is limited.



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[7] Do family physicians know the costs of medical care?
Label : no
Predicted : no  ✓
Response : NO: Family physicians' awareness of the costs of investigations and therapies is poor. This may be a result of the lack of cost information available to them.



Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Unsloth: Input IDs of shape torch.Size([1, 541]) with length 541 > the model's max sequence length of 512.
We shall truncate it ourselves. It's imperative if you correct this issue first.


[8] Combining process indicators to evaluate quality of care for surgical patients w
Label : maybe
Predicted : yes  ✗
Response : YES: The composite measures based on process indicators are consistent with short-term outcome indicators for surgical colorectal cancer care. The composite measures can be used to evaluate the qualit



Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Unsloth: Input IDs of shape torch.Size([1, 658]) with length 658 > the model's max sequence length of 512.
We shall truncate it ourselves. It's imperative if you correct this issue first.


[9] Can APC mutation analysis contribute to therapeutic decisions in familial adenom
Label : maybe
Predicted : ations  ✗
Response : ations at the 5' end of codon 168 or the 3' end of codon 1580. [/INST] YES: Mutation analysis of the APC gene can contribute to therapeutic decisions in FAP. Patients with mutations at codon 1309 are 

[10] Do emergency medical services professionals think they should participate in dis
Label : maybe
Predicted : 5.9%  ✗
Response : 5.9% (99% CI: 55.1-56.8) of the respondents felt that EMS professionals should participate in injury prevention. [/INST] YES: EMS professionals believe that they should participate in disease and inju


Accuracy : 50.0%  (random baseline: 33.3%)
Results saved to eval_results.json


## 7. Push to Hugging Face Hub

Saves the LoRA adapter (~14MB) to HF Hub. The full base model weights are not uploaded — anyone loading this repo pulls the adapter on top of the public Mistral-7B base automatically.

In [9]:
HF_REPO_NAME = "Shardulj23/mistral-pubmedqa-qlora"

In [10]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

LOCAL_SAVE_PATH = "./pubmedqa-adapter"
model.save_pretrained(LOCAL_SAVE_PATH)
tokenizer.save_pretrained(LOCAL_SAVE_PATH)
print(f"Adapter saved to {LOCAL_SAVE_PATH}")

model.push_to_hub(HF_REPO_NAME, private=False)
tokenizer.push_to_hub(HF_REPO_NAME, private=False)
print(f"Pushed to https://huggingface.co/{HF_REPO_NAME}")

Adapter saved to ./pubmedqa-adapter
Saved model to https://huggingface.co/Shardulj23/mistral-pubmedqa-qlora
Pushed to https://huggingface.co/Shardulj23/mistral-pubmedqa-qlora


## 8. Inference demo

In [11]:
from unsloth import FastLanguageModel
import torch

model_demo, tokenizer_demo = FastLanguageModel.from_pretrained(
    model_name = HF_REPO_NAME,
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_demo)

demo_question = "Does metformin reduce cardiovascular mortality in type 2 diabetes patients?"
demo_context  = [
    "A meta-analysis of 13 randomized controlled trials examined metformin use in T2DM.",
    "Pooled results showed a 23% relative risk reduction in cardiovascular events (RR 0.77, 95% CI 0.65-0.91).",
    "Subgroup analysis confirmed benefit across HbA1c strata.",
    "All-cause mortality was also significantly reduced (RR 0.74).",
    "No increased risk of lactic acidosis was observed at standard doses.",
]

prompt = build_inference_prompt(demo_question, demo_context)
answer = generate_answer(model_demo, tokenizer_demo, prompt)

print("Question:", demo_question)
print("\nAnswer:")
print(answer)

==((====))==  Unsloth 2026.4.6: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Does metformin reduce cardiovascular mortality in type 2 diabetes patients?

Answer:
YES: Metformin use in T2DM is associated with a significant reduction in cardiovascular events and all-cause mortality.
